# Optimisation

In [7]:
from enum import Enum
from typing import Callable, Tuple

from autograd.extend import primitive, defvjp
from autograd import numpy as npa
from autograd import tensor_jacobian_product
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.pyplot as plt
import meep as mp
import meep.adjoint as mpa
import numpy as np
import math
from scipy.interpolate import RegularGridInterpolator

## Paramètres

In [8]:
# --- Paramètres physiques ---
resolution = 100         # pixels/um 
lam_cen = 1.55             # longueur d'onde centrale (um)
theta = math.radians(10)   # angle d'incidence de la fibre par rapport a la normale (rad)
n_core = np.sqrt(12.0)     # indice de refraction du silicium 
Si = mp.Medium(epsilon=12.0)  # materiau silicium 
h = 0.22                   # epaisseur totale du slab silicium (um)
gp = 0.890                 # periode du reseau (um)
gdc = 0.5                  # duty cycle du reseau (fraction de silicium par periode)
gN = 5                    # nombre de dents du reseau
t_slab = 0.07              # hauteur de la partie non gravée du réseau

# --- Domaine ---
dpml = 1.0                 # epaisseur des couches PML (um)
margin_before = 8.0        # marge d'air avant le reseau
dair = 2.0                 # marge d'air au-dessus et en dessous du slab, en um
Lwg = 8.0                  # longueur du guide de sortie monomode (um)
Lgrating = gN * gp         # longueur totale du reseau
sx = dpml + margin_before + Lgrating + Lwg + dpml   # largeur totale du domaine de simulation 
sy = dpml + dair + h + dair + dpml                   # hauteur totale du domaine de simulation 
cell_size = mp.Vector3(sx, sy, 0)                    # taille de la cellule Meep
x0 = -0.5 * sx + dpml + margin_before                # position x de debut du reseau (bord gauche de la 1ere dent)
x_wg_start = x0 + Lgrating                           # position x ou commence le guide de sortie (juste apres le reseau)
boundary_layers = [mp.PML(dpml)]                     # conditions aux limites absorbantes sur les 4 bords du domaine
fcen = 1 / lam_cen                                   # frequence centrale correspondant a lam_cen (unites Meep : c=1)

# --- Source ---
d_standoff = 1.0            # distance verticale entre la ligne source et la surface du reseau (um)
y_src = h/2 + d_standoff     # position y de la source 
x_focus = x0 + Lgrating/2    # position x du point focal du faisceau gaussien 
beam_kdir = mp.Vector3(math.sin(theta), -math.cos(theta), 0)  # vecteur direction de propagation du faisceau 
beam_x0 = mp.Vector3(0, -d_standoff, 0)              # position du point focal
df = 0.05 * fcen             # largeur de bande de la source pulsee 

# --- Variables ---
t_etch = h - t_slab          # hauteur de la partie gravée du réseau
mon_pt = mp.Vector3(x_wg_start + 0.7 * Lwg, 0, 0)

# --- Design Region ---
design_region_resolution = 10 * resolution
design_region_size_x = 20 * 0.72 # 14.4 um
design_region_size_y = h
design_region_size = mp.Vector3(
    design_region_size_x, design_region_size_y, 0
)
nx_design_grid = int(design_region_size.x * design_region_resolution) + 1
ny_design_grid = int(design_region_size.y * design_region_resolution) + 1
nx_sim_grid = int(design_region_size.x * resolution) + 1
ny_sim_grid = int(design_region_size.y * resolution) + 1

## Script

In [47]:
def design_region_to_meshgrid(nx: int, ny: int) -> Tuple[np.ndarray, np.ndarray]: # 2
    """Returns the 2D coordinates of the meshgrid for the design region.

    Args:
      nx, ny: the number of grid points in the x and y directions, respectively.

    Returns:
      The coordinates of the x and y grid points (2D arrays) as a 2-tuple.
    """

    xcoord = np.linspace(-0.5 * design_region_size.x, +0.5 * design_region_size.x, nx)
    ycoord = np.linspace(-0.5 * design_region_size.y, +0.5 * design_region_size.y, ny)
    xv, yv = np.meshgrid(xcoord, ycoord, indexing="ij")

    return xv, yv

@primitive
def teeth_weight(params): # 3
    gp, gdc = params
    xv, yv = design_region_to_meshgrid(nx_sim_grid, ny_sim_grid)
    x_loc = xv - xv[0][0]                    # shift the origine at the start of the grid instead of the middle of the grid
    x_mod = np.mod(x_loc, gp)                # position in the period
    weights = np.where(x_mod <= gdc * gp, 1, 0)
    return weights.flatten()

def teeth_weight_vjp(ans, params):
    gp, gdc = params
    eps = 1.0 / resolution   # perturbation

    w0 = teeth_weight([gp, gdc])
    w_dgp = teeth_weight([gp + eps, gdc])
    w_dgdc = teeth_weight([gp, gdc + eps])

    jac_gp = (w_dgp - w0) / eps      # d(rho)/d(gp) per pixel
    jac_gdc = (w_dgdc - w0) / eps    # d(rho)/d(gdc) per pixel

    def vjp(g):
        # g : gradient adjoint 
        dF_dgp = np.tensordot(g, jac_gp, axes=1)
        dF_dgdc = np.tensordot(g, jac_gdc, axes=1)
        return np.array([dF_dgp, dF_dgdc])

    return vjp

defvjp(teeth_weight, teeth_weight_vjp)

fake_grad_pixels = np.random.rand(nx_sim_grid * ny_sim_grid)  # un faux gradient par pixel, juste pour tester la mecanique

grad_params_test = tensor_jacobian_product(teeth_weight, 0)(
    np.array([gp, gdc]), fake_grad_pixels
)
print("gradient teste par rapport a [gp, gdc] :", grad_params_test)

gradient teste par rapport a [gp, gdc] : [-12208.97355923  16784.26498326]


In [48]:
from enum import Enum
from typing import Callable, Tuple

from autograd.extend import primitive, defvjp
from autograd import numpy as npa
from autograd import tensor_jacobian_product
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.pyplot as plt
import meep as mp
import meep.adjoint as mpa
import numpy as np
import math
from scipy.interpolate import RegularGridInterpolator

# --- Variables ---
## --- Paramètres physiques ---
resolution = 100         # pixels/um 
lam_cen = 1.55             # longueur d'onde centrale (um)
theta = math.radians(10)   # angle d'incidence de la fibre par rapport a la normale (rad)
n_core = np.sqrt(12.0)     # indice de refraction du silicium 
Si = mp.Medium(epsilon=12.0)  # materiau silicium 
Air = mp.Medium(index=1.0)  # materieau air
h = 0.22                   # epaisseur totale du slab silicium (um)
gp = 0.890                 # periode du reseau (um)
gdc = 0.5                  # duty cycle du reseau (fraction de silicium par periode)
gN = 5                    # nombre de dents du reseau
t_slab = 0.07              # hauteur de la partie non gravée du réseau
t_etch = h - t_slab          # hauteur de la partie gravée du réseau

## --- Domaine ---
dpml = 1.0                 # epaisseur des couches PML (um)
margin_before = 8.0        # marge d'air avant le reseau
dair = 2.0                 # marge d'air au-dessus et en dessous du slab, en um
Lwg = 8.0                  # longueur du guide de sortie monomode (um)
Lgrating = gN * gp         # longueur totale du reseau
sx = dpml + margin_before + Lgrating + Lwg + dpml   # largeur totale du domaine de simulation 
sy = dpml + dair + h + dair + dpml                   # hauteur totale du domaine de simulation 
cell_size = mp.Vector3(sx, sy, 0)                    # taille de la cellule Meep
x0 = -0.5 * sx + dpml + margin_before                # position x de debut du reseau (bord gauche de la 1ere dent)
x_wg_start = x0 + Lgrating                           # position x ou commence le guide de sortie (juste apres le reseau)
boundary_layers = [mp.PML(dpml)]                     # conditions aux limites absorbantes sur les 4 bords du domaine
fcen = 1 / lam_cen                                   # frequence centrale correspondant a lam_cen (unites Meep : c=1)
mon_pt = mp.Vector3(x_wg_start + 0.7 * Lwg, 0, 0)

## --- Source ---
d_standoff = 1.0            # distance verticale entre la ligne source et la surface du reseau (um)
y_src = h/2 + d_standoff     # position y de la source 
x_focus = x0 + Lgrating/2    # position x du point focal du faisceau gaussien 
beam_kdir = mp.Vector3(math.sin(theta), -math.cos(theta), 0)  # vecteur direction de propagation du faisceau 
beam_x0 = mp.Vector3(0, -d_standoff, 0)              # position du point focal
df = 0.05 * fcen             # largeur de bande de la source pulsee 

## --- Design Region ---
design_region_resolution = 10 * resolution
design_region_size_x = 20 * 0.72 # 14.4 um
design_region_size_y = h
design_region_size = mp.Vector3(
    design_region_size_x, design_region_size_y, 0
)
nx_design_grid = int(design_region_size.x * design_region_resolution) + 1
ny_design_grid = int(design_region_size.y * design_region_resolution) + 1
nx_sim_grid = int(design_region_size.x * resolution) + 1
ny_sim_grid = int(design_region_size.y * resolution) + 1

# --- Fonctions ---

DEBUG_OUTPUT = False

def design_region_to_meshgrid(nx: int, ny: int) -> Tuple[np.ndarray, np.ndarray]: # 2
    """Returns the 2D coordinates of the meshgrid for the design region.

    Args:
      nx, ny: the number of grid points in the x and y directions, respectively.

    Returns:
      The coordinates of the x and y grid points (2D arrays) as a 2-tuple.
    """

    xcoord = np.linspace(-0.5 * design_region_size.x, +0.5 * design_region_size.x, nx)
    ycoord = np.linspace(-0.5 * design_region_size.y, +0.5 * design_region_size.y, ny)
    xv, yv = np.meshgrid(xcoord, ycoord, indexing="ij")

    return xv, yv

@primitive
def teeth_weight(params): # 3
    gp, gdc = params
    xv, yv = design_region_to_meshgrid(nx_sim_grid, ny_sim_grid)
    x_loc = xv - xv[0][0]                    # shift the origine at the start of the grid instead of the middle of the grid
    x_mod = np.mod(x_loc, gp)                # position in the period
    weights = np.where(x_mod <= gdc * gp, 1, 0)
    return weights.flatten()

def teeth_weights_vjp(ans, params):
    gp, gdc = params
    eps = 1.0 / resolution   # perturbation

    w0 = teeth_weights([gp, gdc])
    w_dgp = teeth_weights([gp + eps, gdc])
    w_dgdc = teeth_weights([gp, gdc + eps])

    jac_gp = (w_dgp - w0) / eps      # d(rho)/d(gp) per pixel
    jac_gdc = (w_dgdc - w0) / eps    # d(rho)/d(gdc) per pixel

    def vjp(g):
        # g : gradient adjoint 
        dF_dgp = np.tensordot(g, jac_gp, axes=1)
        dF_dgdc = np.tensordot(g, jac_gdc, axes=1)
        return np.array([dF_dgp, dF_dgdc])

    return vjp

defvjp(teeth_weight, teeth_weight_vjp)

def build_sim(params):
    gp, gdc = params

    # --- Variables ---
    Lgrating = gN * gp
    sx_local = dpml + margin_before + Lgrating + Lwg + dpml
    cell_size_local = mp.Vector3(sx_local, sy, 0)
    x0_local = -0.5 * sx_local + dpml + margin_before
    x_wg_start_local = x0_local + Lgrating
    x_focus_local = x0_local + Lgrating / 2

    # --- Design region ---
    design_variables = mp.MaterialGrid(
        mp.Vector3(nx_sim_grid, ny_sim_grid), Air, Si,
        weights=np.ones((nx_sim_grid, ny_sim_grid)),
        do_averaging=False,
    )
    design_region = mpa.DesignRegion(
        design_variables,
        volume=mp.Volume(
            center=mp.Vector3(x0_local + design_region_size_x/2, -h/2 + t_slab + t_etch/2, 0),
            size=design_region_size,
        ),
    )

    # --- Source ---
    sources = [mp.Source(
        src=mp.GaussianSource(fcen, fwidth=df),
        component=mp.Ez,
        center=mp.Vector3(x_focus_local, y_src, 0),
        size=mp.Vector3(20, 0, 0),
    )]

    # --- Geometrie ---
    geometry = [
        mp.Block(
            size=mp.Vector3(Lgrating + Lwg + dpml, t_slab, mp.inf),
            center=mp.Vector3(x0_local + (Lgrating+Lwg+dpml)/2, -h/2 + t_slab/2, 0),
            material=Si,
        ),
        mp.Block(
            size=mp.Vector3(Lwg + dpml, t_etch, mp.inf),
            center=mp.Vector3(x_wg_start_local + (Lwg+dpml)/2, -h/2 + t_slab + t_etch/2, 0),
            material=Si,
        ),
        mp.Block(
            center=design_region.center,
            size=design_region.size,
            material=design_variables,
        ),
    ]

    sim = mp.Simulation(
        resolution=resolution,
        cell_size=cell_size_local,
        boundary_layers=boundary_layers,
        sources=sources,
        geometry=geometry,
    )

    mon_pt_local = mp.Vector3(x_wg_start_local + 0.7 * Lwg, 0, 0)

    obj_args = [mpa.EigenmodeCoefficient(
        sim, mp.Volume(center=mon_pt_local, size=mp.Vector3(0, sy - 2*dpml, 0)), mode=1,
        eig_parity=mp.ODD_Z + mp.EVEN_Y,
    )]

    def obj_func(mode_coeff):
        return npa.abs(mode_coeff) ** 2

    opt = mpa.OptimizationProblem(
        simulation=sim,
        objective_functions=obj_func,
        objective_arguments=obj_args,
        design_regions=[design_region],
        frequencies=[fcen],
    )

    return opt

opt = build_sim(np.array([gp, gdc]))
weights = teeth_weight(np.array([gp, gdc]))
val, grad_pixels = opt([weights], need_gradient=True)
print("val =", val)
print("grad_pixels.shape =", grad_pixels.shape, " attendu:", nx_sim_grid*ny_sim_grid)

Starting forward run...
-----------
Initializing structure...
time for choose_chunkdivision = 0.00341916 s
Working in 2D dimensions.
Computational cell is 22.45 x 6.22 x 0 with resolution 100
     block, center = (4.5,-0.075,0)
          size (13.45,0.07,1e+20)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (12,12,12)
     block, center = (6.725,0.035,0)
          size (9,0.15,1e+20)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (12,12,12)
     block, center = (4.975,0.035,0)
          size (14.4,0.22,0)
          axes (1,0,0), (0,1,0), (0,0,1)
time for set_epsilon = 2.28877 s
-----------
on time step 415 (time=2.075), 0.00965592 s/step
on time step 856 (time=4.28), 0.00907745 s/step
on time step 1282 (time=6.41), 0.0094015 s/step
on time step 1729 (time=8.645), 0.00896585 s/step
on time step 2177 (time=10.885), 0.00894825 s/step
on time step 2627 (time=13.135), 0.00890926 s/step
on time step 3086 (t

run 0 finished at t = 404.3 (80860 timesteps)
MPB solved for frequency_1(0.645161,0,0) = 0.323742 after 8 iters
MPB solved for frequency_1(1.59482,0,0) = 0.601011 after 7 iters
MPB solved for frequency_1(1.72602,0,0) = 0.643685 after 5 iters
MPB solved for frequency_1(1.73042,0,0) = 0.645155 after 4 iters
MPB solved for frequency_1(1.73044,0,0) = 0.645161 after 2 iters
MPB solved for frequency_1(1.73044,0,0) = 0.645161 after 1 iters
Dominant planewave for band 1: (1.730439,-0.000000,0.000000)
Starting adjoint run...


MPB solved for frequency_1(-0.645161,0,0) = 0.323742 after 8 iters
MPB solved for frequency_1(-1.59482,0,0) = 0.601011 after 7 iters
MPB solved for frequency_1(-1.72602,0,0) = 0.643685 after 5 iters
MPB solved for frequency_1(-1.73042,0,0) = 0.645155 after 4 iters
MPB solved for frequency_1(-1.73044,0,0) = 0.645161 after 2 iters
MPB solved for frequency_1(-1.73044,0,0) = 0.645161 after 1 iters
on time step 456 (time=2.28), 0.0087831 s/step
on time step 909 (time=4.545), 0.00883048 s/step
on time step 1369 (time=6.845), 0.00870767 s/step
on time step 1824 (time=9.12), 0.00880547 s/step
on time step 2283 (time=11.415), 0.00871481 s/step
on time step 2744 (time=13.72), 0.008685 s/step
on time step 3206 (time=16.03), 0.0086754 s/step
on time step 3665 (time=18.325), 0.00873168 s/step
on time step 4135 (time=20.675), 0.00851723 s/step
on time step 4572 (time=22.86), 0.00916248 s/step
on time step 4971 (time=24.855), 0.0100419 s/step
on time step 5428 (time=27.14), 0.00876947 s/step
on time 

In [49]:
grad_params = tensor_jacobian_product(teeth_weight, 0)(
    np.array([gp, gdc]), grad_pixels
)
print("dF/dgp =", grad_params[0], " dF/dgdc =", grad_params[1])

dF/dgp = -2022.6228394817585  dF/dgdc = 240.97973267652915
